In [1]:
import dolfinx
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver, self_supervised_train

from dolfinx.io import XDMFFile
from mpi4py import MPI
from SPDE_problems import int_to_prblm

tset_dir = "training_set"
tset = graph_dataset(f"data/{tset_dir}/input_values")

class batched_loss_fn():
    def __init__(self, set):
        self.fsl = {}
        for G in set:
            num = G.mesh_id[0]
            with XDMFFile(MPI.COMM_WORLD, f"data/training_set/mesh_files/mesh_{G.mesh_id[0]}.xdmf", "r") as xdmf:
                mesh = xdmf.read_mesh(name="mesh")


            fs = int_to_prblm(idx=G.prblm_id, mesh=mesh)
            self.fsl[int(G.mesh_id)] = fem_solver(fs)

    def __call__(self, ptr, idx, y):
        loss_vals = [self.fsl[int(idx[i])](y[ptr[i]:ptr[i+1]] ) for i in range(len(ptr)-1)]
        return torch.stack(loss_vals).sum()
    
loss_fn = batched_loss_fn(tset)


batch_size = 15
loader = train_loader(batch_size=batch_size, set=tset)
model=ClampRestriction(GATv2)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10


In [2]:

for i in range(2000):
    loss = self_supervised_train(model=model, loader=loader,loss_fn=loss_fn, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/clamp_GATv2.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        #scheduler.step(loss)

        


iteration 0: new loss: 0.4225396358191198
iteration 1: new loss: 0.40917814982482886
iteration 2: new loss: 0.4019274866428685
iteration 3: new loss: 0.39702391059687253
iteration 4: new loss: 0.3885269835289942
iteration 5: 0.7829356020615041
iteration 6: 0.38884051373847534
iteration 7: new loss: 0.386098635188392
iteration 8: new loss: 0.38400857972953645
iteration 9: new loss: 0.3825227217170511
iteration 10: new loss: 0.3814631910152167
iteration 11: new loss: 0.3810504096344354
iteration 12: new loss: 0.3808268369736503
iteration 13: new loss: 0.38069465131202124
iteration 14: new loss: 0.38060465189471754
iteration 15: new loss: 0.38047242505507645
iteration 16: 0.380539243260613
iteration 17: new loss: 0.38029378181454115
iteration 18: 0.3802956740024009
iteration 19: new loss: 0.38008263305121875
iteration 20: 0.3802093855996875
iteration 21: 0.3801618174161876
iteration 22: 0.38016441943966334
iteration 23: 0.3801628093562094
iteration 24: 0.3802351428513399
iteration 25: new

KeyboardInterrupt: 